In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split

# 1. Load IMDB dataset
vocab_size = 10000
max_length = 200

data = pd.read_csv('IMDB Dataset.csv')

X = data['review'].astype(str).values
y = data['sentiment'].map({'positive': 1, 'negative': 0}).values

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X)

X = tokenizer.texts_to_sequences(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Pad sequences
X_train = pad_sequences(X_train, maxlen=max_length)
X_test = pad_sequences(X_test, maxlen=max_length)

# 3. Build RNN model
model = Sequential([
    Input(shape=(max_length,)),
    Embedding(vocab_size, 32),
    SimpleRNN(32),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 4. Train model
model.fit(X_train, y_train, epochs=5,
          batch_size=128, validation_split=0.2)

# 5. Evaluate model
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print("\nTEST RESULTS")
print("Test Accuracy:", round(accuracy * 100, 2), "%")
print("Test Loss:", round(loss, 4))

# 6. Predict a review
word_index = tokenizer.word_index

review = "This movie was excellent and amazing. I really enjoyed it."

sequence = tokenizer.texts_to_sequences([review])
sequence = pad_sequences(sequence, maxlen=max_length)

prediction = model.predict(sequence, verbose=0)[0][0]

print("\nPREDICTION")
print("Review:", review)
print("Prediction:", "Positive" if prediction >= 0.5 else "Negative")

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 23s 75ms/step - accuracy: 0.6321 - loss: 0.6266 - val_accuracy: 0.8021 - val_loss: 0.4393
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 67ms/step - accuracy: 0.8328 - loss: 0.3909 - val_accuracy: 0.8296 - val_loss: 0.4040
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - accuracy: 0.8898 - loss: 0.2773 - val_accuracy: 0.8388 - val_loss: 0.3945
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 16s 64ms/step - accuracy: 0.9327 - loss: 0.1860 - val_accuracy: 0.8254 - val_loss: 0.4446
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 21s 65ms/step - accuracy: 0.9660 - loss: 0.1059 - val_accuracy: 0.8326 - val_loss: 0.4874

TEST RESULTS
Test Accuracy: 83.6 %
Test Loss: 0.4786

PREDICTION
Review: This movie was excellent and amazing. I really enjoyed it.
Prediction: Positive
